In [0]:
import sys, os
sys.path.append(os.path.abspath(".."))
from src.common.config import PipelineConfig
from src.common.observabilidad import aplicar

cfg = PipelineConfig(fecha_corte="2026-11-05")
print("Vistas creadas:", aplicar(spark, cfg))

for v in ["vw_obs_ejecuciones", "vw_obs_volumenes", "vw_obs_frescura",
          "vw_obs_duplicados", "vw_obs_esquemas", "vw_obs_calidad"]:
    spark.sql(f"GRANT SELECT ON TABLE workspace.default.{v} TO `grp_auditoria`")
print("Permisos de lectura otorgados a grp_auditoria")

In [0]:
%sql
SELECT run_id, fecha_corte, estado_ejecucion, estado_bronze, estado_silver, estado_gold,
       duracion_min, costo_estimado_usd, gold_publicado_col, sla, error
FROM workspace.default.vw_obs_ejecuciones
ORDER BY inicio_ts DESC;

In [0]:
%sql
SELECT * FROM workspace.default.vw_obs_frescura;

In [0]:
%sql
SELECT capa, tabla, entrada, aceptados, rechazados, pct_rechazo
FROM workspace.default.vw_obs_volumenes
QUALIFY ROW_NUMBER() OVER (PARTITION BY tabla ORDER BY registrado_ts DESC) = 1
ORDER BY capa, tabla;